# Marcar remates por color — `.docx` → `.md`

Convierte documentos del **histórico de chistes** (con remates marcados en rojo) a
Markdown con etiquetas embebidas, según la spec §7 (decisión P15):

| Color de fuente | Etiqueta | Semántica |
|---|---|---|
| `#FF0000` (rojo puro) | `[REMATE]…[/REMATE]` | Remate principal — **cierra** el chiste |
| `#980000` (burdeos) | `[CHISTOIDE]…[/CHISTOIDE]` | Mini-remate interno — **no** es frontera |

**Reglas:** clasificación por tono con margen (no hex exacto) · fusión de runs contiguos
del mismo color · un span puede cruzar párrafos · espacios/puntuación fuera de las
etiquetas · cubre tablas, hyperlinks y listas · **validación round-trip obligatoria**
(si los caracteres de color no cuadran con los etiquetados, el fichero NO se genera).

**Uso:**
1. Exporta tus Google Docs como **`.docx`** a una carpeta de Drive (Archivo → Descargar → .docx). Los `.gdoc` nativos no sirven.
2. Ajusta `CARPETA_DOCS` en la celda de configuración.
3. Runtime → Run all. Los `.md` aparecen en `CARPETA_SALIDA`. El original nunca se toca.

_Sin instalación: solo librería estándar de Python._
_Limitación documentada: se lee el color directo del run (determinista); colores heredados de estilos de párrafo no se consideran._


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path


In [ ]:
# --- Rutas ---
CARPETA_DOCS   = "/content/drive/MyDrive/Chistes_Historico/"   # .docx de entrada
CARPETA_SALIDA = os.path.join(CARPETA_DOCS, "Markdown")        # .md de salida
SOBRESCRIBIR   = True   # False = no regenerar .md ya existentes

# --- Clasificación de color por tono (margen, no igualdad exacta de hex) ---
UMBRAL_GB        = 90    # G y B por debajo de esto -> candidato a rojo
UMBRAL_ROJO_MIN  = 120   # R minimo para considerarse rojo
UMBRAL_ROJO_PURO = 200   # R >= esto -> REMATE (#FF0000); si no -> CHISTOIDE (#980000)

# --- Constantes tecnicas ---
W = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'
# Caracteres que quedan FUERA de las etiquetas cuando caen en el borde de un span
BORDES = ' \t\n.,;:!?\u2026\u00b7\u2014\u2013-"\'\u201c\u201d\u2018\u2019\u00ab\u00bb()[]{}\u00a1\u00bf'


In [ ]:
def clasificar_color(hex_val):
    """Clasifica un color de fuente por tono: 'REMATE' | 'CHISTOIDE' | None.

    #FF0000 (rojo puro)  -> REMATE     (cierra el chiste)
    #980000 (burdeos)    -> CHISTOIDE  (mini-remate interno, no es frontera)
    Cualquier otro color -> None       (texto normal)
    """
    if not hex_val or hex_val.lower() == 'auto':
        return None
    try:
        r = int(hex_val[0:2], 16)
        g = int(hex_val[2:4], 16)
        b = int(hex_val[4:6], 16)
    except ValueError:
        return None
    if g < UMBRAL_GB and b < UMBRAL_GB and r >= UMBRAL_ROJO_MIN:
        return 'REMATE' if r >= UMBRAL_ROJO_PURO else 'CHISTOIDE'
    return None


In [ ]:
def extraer_parrafos(ruta_docx):
    """Lee word/document.xml y devuelve una lista de parrafos.

    Cada parrafo es una lista de tokens (etiqueta, texto), donde etiqueta es
    'REMATE' | 'CHISTOIDE' | None. Recorre TODOS los parrafos del documento
    (incluidos los de dentro de tablas) y TODOS los runs de cada parrafo
    (incluidos los de dentro de hyperlinks). w:tab -> tabulador, w:br -> salto.
    """
    with zipfile.ZipFile(ruta_docx) as z:
        root = ET.fromstring(z.read('word/document.xml'))

    parrafos = []
    for p in root.iter(f'{W}p'):            # incluye parrafos dentro de w:tbl
        tokens = []
        for r in p.iter(f'{W}r'):           # incluye runs dentro de w:hyperlink
            color = None
            rpr = r.find(f'{W}rPr')
            if rpr is not None:
                c = rpr.find(f'{W}color')
                if c is not None:
                    color = c.get(f'{W}val')
            partes = []
            for hijo in r:
                if hijo.tag == f'{W}t':
                    partes.append(hijo.text or '')
                elif hijo.tag == f'{W}tab':
                    partes.append('\t')
                elif hijo.tag == f'{W}br':
                    partes.append('\n')
            texto = ''.join(partes)
            if texto:
                tokens.append((clasificar_color(color), texto))
        parrafos.append(tokens)
    return parrafos


In [ ]:
def fusionar_tokens(tokens):
    """Fusiona tokens contiguos con la misma etiqueta en un solo segmento.
    Tambien absorbe separadores de solo-espacios entre dos tramos iguales:
    [REMATE]('toma') + (' ') + [REMATE]('ya')  ->  [REMATE]('toma ya')."""
    segs = []
    for et, tx in tokens:
        if segs and segs[-1][0] == et:
            segs[-1][1] += tx
        else:
            segs.append([et, tx])
    i = 0
    while i + 2 < len(segs):
        a, b, c = segs[i], segs[i + 1], segs[i + 2]
        if a[0] is not None and a[0] == c[0] and b[0] is None and not b[1].strip():
            a[1] += b[1] + c[1]
            del segs[i + 1:i + 3]
        else:
            i += 1
    return segs


def emitir_span(etiqueta, texto):
    """Envuelve texto en [ETIQUETA]...[/ETIQUETA] dejando fuera los espacios
    y la puntuacion de los bordes. Si el nucleo queda vacio, no etiqueta."""
    ini, fin = 0, len(texto)
    while ini < fin and texto[ini] in BORDES:
        ini += 1
    while fin > ini and texto[fin - 1] in BORDES:
        fin -= 1
    nucleo = texto[ini:fin]
    if not nucleo:
        return texto
    return f"{texto[:ini]}[{etiqueta}]{nucleo}[/{etiqueta}]{texto[fin:]}"


def construir_markdown(parrafos):
    """Serializa los parrafos a Markdown con las etiquetas embebidas.

    Un span del mismo color que termina un parrafo y abre el siguiente se
    mantiene como UN SOLO tramo (el salto de parrafo queda dentro). Un parrafo
    vacio entre medias SI cierra el span (decision determinista)."""
    plano = []
    for tokens in parrafos:
        segs = fusionar_tokens(tokens)
        if not any(tx.strip() for _, tx in segs):
            segs = []                       # parrafo vacio
        if plano:
            plano.append([None, '\n\n'])    # separador de parrafo
        plano.extend(segs)

    # span multi-parrafo: [T] + separador + [T] -> un solo segmento
    i = 0
    while i + 2 < len(plano):
        a, b, c = plano[i], plano[i + 1], plano[i + 2]
        if a[0] is not None and a[0] == c[0] and b[0] is None and not b[1].strip():
            a[1] += b[1] + c[1]
            del plano[i + 1:i + 3]
        else:
            i += 1

    md = ''.join(emitir_span(et, tx) if et else tx for et, tx in plano)
    md = re.sub(r'\n{3,}', '\n\n', md).strip() + '\n'
    return md


In [ ]:
def contar_alfanum(texto):
    return len(re.findall(r'\w', texto))


def validar_roundtrip(parrafos, md):
    """Guardia de integridad: los caracteres alfanumericos de cada color en el
    .docx deben ser EXACTAMENTE los que quedaron dentro de su etiqueta en el .md.
    (La puntuacion/espacios movidos fuera no son alfanumericos, no afectan.)"""
    esperado = {'REMATE': 0, 'CHISTOIDE': 0}
    for tokens in parrafos:
        for et, tx in tokens:
            if et in esperado:
                esperado[et] += contar_alfanum(tx)
    obtenido = {}
    for et in esperado:
        dentro = re.findall(rf'\[{et}\](.*?)\[/{et}\]', md, re.DOTALL)
        obtenido[et] = contar_alfanum(''.join(dentro))
    return esperado == obtenido, esperado, obtenido


In [ ]:
def procesar_docx(ruta_docx, carpeta_salida):
    """Procesa un .docx -> .md con etiquetas. El original NUNCA se toca.
    Si la validacion round-trip falla, lanza error y NO escribe salida."""
    parrafos = extraer_parrafos(ruta_docx)
    md = construir_markdown(parrafos)
    ok, esperado, obtenido = validar_roundtrip(parrafos, md)
    if not ok:
        raise ValueError(
            f"round-trip FALLIDO (esperado {esperado}, obtenido {obtenido}) "
            f"- posible run perdido; no se escribe el .md"
        )
    ruta_md = Path(carpeta_salida) / (Path(ruta_docx).stem + '.md')
    if ruta_md.exists() and not SOBRESCRIBIR:
        return ruta_md, 'omitido (ya existe)'
    ruta_md.write_text(md, encoding='utf-8')
    return ruta_md, (f"OK - REMATE {esperado['REMATE']} chars, "
                     f"CHISTOIDE {esperado['CHISTOIDE']} chars")


In [ ]:
# Bucle de procesamiento: todos los .docx de la carpeta
os.makedirs(CARPETA_SALIDA, exist_ok=True)

for archivo in sorted(os.listdir(CARPETA_DOCS)):
    if not archivo.lower().endswith('.docx') or archivo.startswith('~$'):
        continue
    ruta = os.path.join(CARPETA_DOCS, archivo)
    print(f"\n--- Procesando: {archivo} ---")
    try:
        ruta_md, msg = procesar_docx(ruta, CARPETA_SALIDA)
        print(f"\u2713 {ruta_md.name}: {msg}")
    except Exception as e:
        print(f"X Error procesando {archivo}: {e}")

print("\n--- PROCESO FINALIZADO ---")


In [ ]:
# Vista previa del primer .md generado
salidas = sorted(Path(CARPETA_SALIDA).glob('*.md'))
if salidas:
    print(f"Vista previa de: {salidas[0].name}")
    print("=" * 70)
    print(salidas[0].read_text(encoding='utf-8')[:1500])
else:
    print("No hay .md generados todavia.")
